### Exploratory Notebook

In [34]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, IntegerType
from pyspark.sql.functions import from_json, col
from pyspark.sql.functions import regexp_replace


################################################
# Schemas 

# Define the schema for the JSON data
event_schema = StructType([
    StructField("age_of_insured", IntegerType(), True),
    StructField("coverage_amount", DoubleType(), True),
    StructField("customer_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("policy_id", StringType(), True),
    StructField("policy_type", StringType(), True),
    StructField("premium_amount", DoubleType(), True),
    StructField("region", StringType(), True),
])

# Define the schema for the policy_type field
policy_type_schema = StructType([
                StructField("type", StringType(), True),
                StructField("brand", StringType(), True),
])

########################################################
# Load and clean data
     
# Read the JSON data from the specified path and apply the schema
invalid_json_df =   spark \
                    .read\
                    .format("json")\
                    .schema(event_schema) \
                    .load("/Volumes/ageas/bronze/files")

# Fix the invalid JSON in the policy_type field by replacing single quotes with double quotes
valid_json_df = invalid_json_df.withColumn(
    "policy_type",
    regexp_replace(  
        regexp_replace(col("policy_type"), "'", "\""),
        "None", 
        "\"null\""
    )
)

# Replace policy_type string with a valid JSON object
valid_json_df = valid_json_df.withColumn(
    "policy_type_object",
    from_json(col("policy_type"), policy_type_schema)
)


valid_json_df = valid_json_df.selectExpr(
    "age_of_insured",
    "coverage_amount",
    "customer_id",
    "event_timestamp",
    "event_type",
    "policy_id",
    "policy_type_object.type AS policy_type",
    "policy_type_object.brand AS policy_brand",
    "premium_amount",
    "region"
).filter(col("policy_id") == "POL-60927")

display(valid_json_df)

,age_of_insured,coverage_amount,customer_id,event_timestamp,event_type,policy_id,policy_type,policy_brand,premium_amount,region
0,55,29914.22,CUS-39540,2024-03-25T20:25:51.088Z,purchase,POL-60927,null,LifeSecure,581.55,North
